# Microsoft Agent Framework Demo
Please visit https://aka.ms/agentframework for the official code samples

In [ ]:
from agent_framework import ChatAgent, AgentProtocol, AgentThread, HostedMCPTool
from agent_framework.azure import AzureAIAgentClient
from azure.identity.aio import AzureCliCredential
from typing import Any
from IPython.display import display, Markdown

### Basic example

In [ ]:
async def main():
    async with (
        AzureCliCredential() as credential,
        ChatAgent(
            chat_client=AzureAIAgentClient(async_credential=credential),
            instructions="You are good at telling jokes."
        ) as agent,
    ):
        result = await agent.run("Tell me a joke about a pirate.")
        display(Markdown(result.text))

await main()

### Azure AI Agent with Remote MCP Example

In [ ]:
async def handle_approvals_with_thread(query: str, agent: "AgentProtocol", thread: "AgentThread"):
    """Here we let the thread deal with the previous responses, and we just rerun with the approval."""
    from agent_framework import ChatMessage

    result = await agent.run(query, thread=thread, store=True)
    while len(result.user_input_requests) > 0:
        new_input: list[Any] = []
        for user_input_needed in result.user_input_requests:
            print(
                f"User Input Request for function from {agent.name}: {user_input_needed.function_call.name}"
                f" with arguments: {user_input_needed.function_call.arguments}"
            )
            new_input.append(
                ChatMessage(
                    role="user",
                    contents=[user_input_needed.create_response(True)],
                ),
            )
        result = await agent.run(new_input, thread=thread, store=True)
    return result


async def main() -> None:
    """Example showing Hosted MCP tools for a Azure AI Agent."""
    async with (
        AzureCliCredential() as credential,
        AzureAIAgentClient(async_credential=credential) as chat_client,
    ):
        # commenting out azure-ai observability to supress warnings
        # await chat_client.setup_azure_ai_observability()
        agent = chat_client.create_agent(
            name="DocsAgent",
            instructions="You are a helpful assistant that can help with microsoft documentation questions.",
            tools=HostedMCPTool(
                name="Microsoft Learn MCP",
                url="https://learn.microsoft.com/api/mcp",
            ),
        )
        thread = agent.get_new_thread()
        # First query
        query1 = "How do I create an Azure storage account using az cli?"
        display(Markdown(f"**User:** {query1}"))
        result1 = await handle_approvals_with_thread(query1, agent, thread)
        display(Markdown(f"**{agent.name}:** {result1}"))
        display(Markdown("\n---\n"))
        # Second query
        query2 = "What is Microsoft Agent Framework?"
        display(Markdown(f"**User:** {query2}"))
        result2 = await handle_approvals_with_thread(query2, agent, thread)
        display(Markdown(f"**{agent.name}:** {result2}"))

await main()